# V2 — Fine-tuning de Flan-T5-base

Full fine-tuning de Flan-T5-base sobre un subset de 10.000 muestras de CNN/DailyMail.
Objetivo: superar la línea base zero-shot (ROUGE-1 = 24.96) con una configuración
estándar, sin búsqueda de hiperparámetros (eso es V3).

**Configuración:**
- Full fine-tune (todos los parámetros, 250M)
- fp32 + AdamW (T5 es numéricamente inestable en bf16 durante training)
- Batch efectivo = 16 (per_device=8 × grad_accum=2)
- 2 epochs, learning rate 3e-5
- Eval al final de cada epoch con ROUGE-L como métrica de selección

**Decisión de pasar de T5-large a T5-base:** en iteraciones previas se intentó
fine-tunear Flan-T5-large, pero la combinación de tamaño (780M) + inestabilidad
numérica en bf16 en consumer GPUs hacía el training poco viable en una RTX 5070
de 12 GB. Flan-T5-base (250M) cabe cómodamente en fp32, entrena rápido, y además
habilita una comparativa más interesante contra Qwen3-1.7B: un encoder-decoder
casi 7x más pequeño que el decoder-only.


In [1]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
from src.data.loader import load_config, load_cnn_dailymail
from src.data.preprocessing import build_seq2seq_preprocess_fn, tokenize_dataset
from src.models.loader import load_model
from src.training.trainer import build_seq2seq_trainer

print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

cfg = load_config("../config/config.yaml")

CUDA: True | NVIDIA GeForce RTX 5070
VRAM: 12.8 GB


In [2]:
# Load dataset and model. Reduce subset to 10k for a faster first iteration.
cfg["dataset"]["train_subset"] = 10000  # half the data, plenty for V2
cfg["training"]["num_train_epochs"] = 2  # T5 converges fast on summarization

dataset = load_cnn_dailymail(cfg)
loaded = load_model(cfg["models"]["t5"])

preprocess_fn = build_seq2seq_preprocess_fn(
    tokenizer=loaded.tokenizer,
    max_input_length=cfg["models"]["t5"]["max_input_length"],
    max_target_length=cfg["dataset"]["max_target_length"],
    prefix="summarize: ",
)
tokenized = tokenize_dataset(dataset, preprocess_fn)
print(tokenized)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Tokenizing:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})


In [3]:
# T5-base full fine-tune: fits comfortably in fp32 with AdamW, no tricks needed.
trainer = build_seq2seq_trainer(
    loaded=loaded,
    tokenized_dataset=tokenized,
    cfg=cfg,
    output_subdir="v2_t5_base",
    per_device_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    force_fp32=True,
    optim="adamw_torch",
    enable_gradient_checkpointing=False,
)

In [ ]:
# Train! Expect ~1.5–2 hours on RTX 5070
train_result = trainer.train()
trainer.save_model()
trainer.save_metrics("train", train_result.metrics)
print(train_result.metrics)

{'loss': '4.879', 'grad_norm': '1.987', 'learning_rate': '1.176e-05', 'epoch': '0.08'}
{'loss': '4.295', 'grad_norm': '1.877', 'learning_rate': '2.376e-05', 'epoch': '0.16'}
{'loss': '4.137', 'grad_norm': '1.7', 'learning_rate': '2.936e-05', 'epoch': '0.24'}
{'loss': '4.147', 'grad_norm': '1.694', 'learning_rate': '2.803e-05', 'epoch': '0.32'}


In [ ]:
# Evaluate on the test set with the same 200-sample subset as V1 for comparability
from src.evaluation.inference import generate_summaries
from src.evaluation.metrics import compute_rouge

test_subset = dataset["test"].select(range(200))
articles = test_subset["article"]
references = test_subset["highlights"]

# The trainer has loaded the best checkpoint; use it for inference.
preds = generate_summaries(
    loaded,
    articles,
    max_new_tokens=cfg["dataset"]["max_target_length"],
    num_beams=cfg["generation"]["num_beams"],
    batch_size=4,
)

v2_scores = compute_rouge(preds, references)
print("V2 Flan-T5-base fine-tuned:")
print(v2_scores)
print("\nV1 baseline was: rouge1=27.64 | rouge2=9.70 | rougeL=20.18")

In [ ]:
# Save results and qualitative example
import pandas as pd
pd.DataFrame([v2_scores], index=["t5_v2"]).to_csv(
    "../results/tables/v2_t5_finetuned.csv"
)

print("=" * 80)
print(f"ARTICLE:\n{articles[0][:400]}...\n")
print(f"REFERENCE:\n{references[0]}\n")
print(f"T5 V2 PREDICTION:\n{preds[0]}")

## Análisis de V2 — Flan-T5-base fine-tuned

Resultados sobre las 200 muestras de test (mismo subset que V1):

| Métrica     | V1 zero-shot | V2 fine-tuned | Δ      |
|-------------|--------------|---------------|--------|
| ROUGE-1     | 24.96        | **32.92**     | +7.96  |
| ROUGE-2     | 9.03         | **12.57**     | +3.54  |
| ROUGE-L     | 19.17        | **23.25**     | +4.08  |
| ROUGE-Lsum  | 21.64        | **26.97**     | +5.33  |

**El fine-tuning aporta +7.96 ROUGE-1** — una mejora de casi un tercio sobre
el baseline zero-shot. El efecto es particularmente marcado en ROUGE-Lsum (+5.3),
lo que sugiere que el modelo ha aprendido no solo vocabulario relevante sino
también la estructura de oraciones típica del estilo CNN/DailyMail.

**Contexto contra el techo de referencia:**

| Modelo               | ROUGE-1 | ROUGE-2 | ROUGE-L |
|----------------------|---------|---------|---------|
| BART-large-cnn       | 35.12   | 14.54   | 25.54   |
| Pegasus-cnn          | 34.97   | 14.35   | 25.81   |
| **Flan-T5-base V2**  | **32.92** | **12.57** | **23.25** |

Tras fine-tuning, Flan-T5-base queda a **~2.2 puntos de ROUGE-1 del techo**
(BART/Pegasus), partiendo de un gap de ~10 puntos en zero-shot. En otras palabras,
**el fine-tuning recupera aproximadamente el 78% del gap** respecto a modelos
específicamente entrenados para CNN/DailyMail, con solo 10.000 muestras × 2 epochs
frente a los ~280.000 ejemplos que vieron BART y Pegasus durante su entrenamiento
específico.

**Limitación persistente:** la truncación del 73% de los artículos por el límite
de contexto de 512 tokens sigue siendo un techo estructural. Ninguna cantidad de
fine-tuning puede recuperar información que el modelo nunca ve. Este es un punto
interesante de cara a la comparativa con Qwen3, que gracias a su contexto de
1536 tokens solo trunca un 5.6% de los artículos.
